# NumPilot Demo v1：全新环境验收

本 Notebook 从固定 `v0.2.0a1` tag 开始，走完安装、bootstrap、GLM-5.3 迁移和 NPU 实跑。详细说明与排错边界见 `FRESH-NOTEBOOK-QUICKSTART.md`。本流程不接入 ops-nn。

In [ ]:
from pathlib import Path
import json, os, shlex, subprocess

TEST_ROOT = Path(os.environ.get('NUMPILOT_NOTEBOOK_ROOT', str(Path.cwd() / 'numpilot-demo-v1-notebook-run'))).expanduser().resolve()
REPO, VENV = TEST_ROOT / 'numpilot', TEST_ROOT / '.venv'
PYTHON = VENV / 'bin' / 'python'
proxy = os.environ.get('NUMPILOT_HTTP_PROXY') or input('HTTP(S) proxy: ').strip()
wheel_text = os.environ.get('NUMPILOT_ASNUMPY_WHEEL') or input('P0 AsNumpy wheel absolute path: ').strip()
config_text = os.environ.get('NUMPILOT_CONFIG_HOME') or input('Directory containing private llm.json: ').strip()
ASNUMPY_WHEEL = Path(wheel_text).expanduser().resolve()
LLM_CONFIG_HOME = Path(config_text).expanduser().resolve()
ASNUMPY_SHA256 = 'afe845c45d17e7f5cd619831f555fba3c2344296ffe02f1d251017b04cf0282d'
CANN_HOME = Path(os.environ.get('ASCEND_HOME_PATH', '/usr/local/Ascend/cann-9.0.0'))
wheelhouse_text = os.environ.get('NUMPILOT_WHEELHOUSE', '').strip()
WHEELHOUSE = Path(wheelhouse_text).expanduser().resolve() if wheelhouse_text else None
assert not TEST_ROOT.exists(), f'write-once test root exists: {TEST_ROOT}'
assert ASNUMPY_WHEEL.is_file() and (LLM_CONFIG_HOME / 'llm.json').is_file() and CANN_HOME.is_dir()
env = os.environ.copy()
env.update({'http_proxy': proxy, 'https_proxy': proxy, 'ASCEND_HOME_PATH': str(CANN_HOME), 'ASCEND_TOOLKIT_HOME': str(CANN_HOME), 'ASCEND_OPP_PATH': str(CANN_HOME / 'opp'), 'ASCEND_AICPU_PATH': str(CANN_HOME), 'NUMPILOT_CONFIG_HOME': str(LLM_CONFIG_HOME), 'NUMPILOT_STATE_HOME': str(TEST_ROOT / 'state'), 'XDG_CACHE_HOME': str(TEST_ROOT / 'cache')})
env['LD_LIBRARY_PATH'] = ':'.join(['/usr/local/Ascend/driver/lib64', '/usr/local/Ascend/driver/lib64/common', '/usr/local/Ascend/driver/lib64/driver', str(CANN_HOME / 'aarch64-linux/lib64'), str(CANN_HOME / 'lib64/plugin/opskernel'), str(CANN_HOME / 'lib64/plugin/nnengine'), str(CANN_HOME / 'opp/built-in/op_impl/ai_core/tbe/op_tiling/lib/linux/aarch64')])
def run(*args, cwd=None):
    command = [str(item) for item in args]
    print('$', shlex.join(command))
    return subprocess.run(command, cwd=cwd, env=env, check=True, text=True)


In [ ]:
TEST_ROOT.mkdir(parents=True)
run('git', 'clone', '--branch', 'v0.2.0a1', '--depth', '1', 'https://gitcode.com/li-xingHIT/numpilot.git', REPO)
run('git', '-C', REPO, 'status', '--short', '--branch')
run('git', '-C', REPO, 'describe', '--tags', '--exact-match')


In [ ]:
run('python3.10', '-m', 'venv', VENV)
if WHEELHOUSE is not None:
    assert WHEELHOUSE.is_dir(), WHEELHOUSE
    run(PYTHON, '-m', 'pip', '--isolated', 'install', '--no-index', '--find-links', WHEELHOUSE, 'numpilot[demo]==0.2.0a1')
else:
    run(PYTHON, '-m', 'pip', 'install', '-e', f'{REPO}[demo]')
run(PYTHON, '-m', 'numpilot', '--version')
run(PYTHON, '-m', 'numpilot', 'paths', '--json')


In [ ]:
RECEIPT = TEST_ROOT / 'bootstrap-receipt.json'
common = ['--asnumpy-wheel', str(ASNUMPY_WHEEL), '--asnumpy-wheel-sha256', ASNUMPY_SHA256, '--with-cannbot', '--accept-cannbot-license']
run(PYTHON, '-m', 'numpilot', 'bootstrap', '--plan', '--json', *common)
run(PYTHON, '-m', 'numpilot', 'bootstrap', '--apply', *common, '--receipt', RECEIPT)
run(PYTHON, '-m', 'numpilot', 'bootstrap', '--verify', '--with-cannbot', '--accept-cannbot-license')
run(PYTHON, '-m', 'numpilot', 'doctor', '--backend', 'npu')
receipt = json.loads(RECEIPT.read_text(encoding='utf-8'))
assert receipt['status'] == 'ready' and receipt['ready'] is True
assert receipt['wheel']['sha256'] == ASNUMPY_SHA256
assert receipt['verification']['asnumpy']['wheel_file_identity_verified'] is True
print(receipt['wheel']['trust']['profile_id'])


In [ ]:
run(PYTHON, '-m', 'numpilot', 'configure-llm', '--check')
SOURCE = REPO / 'evidence/demotest-case01-20260825/source.py'
OUT = TEST_ROOT / 'case01-migration'
run(PYTHON, '-m', 'numpilot', 'migrate', SOURCE, '--mode', 'script', '--backend', 'npu', '--fallback-policy', 'reject', '--target-soc', 'ascend910b', '--cann-version', '9.0.0', '--out', OUT)
report = json.loads((OUT / 'script-migration-report.json').read_text(encoding='utf-8'))
assert report['status'] == 'migrated_scoped_npu'
assert report['provider'] == 'glm' and report['requested_model'] == report['response_model'] == 'glm-5.3'
assert report['execution_performed'] is False
print(report['migrated_sha256'])


In [ ]:
run(PYTHON, SOURCE)
run(PYTHON, OUT / 'migrated.py')
print('PASS: clone -> install -> bootstrap -> GLM migration -> NPU execution')
